# Costruzione del dataset estesoLa replica usa 17 cuscinetti, un solo regime e frame tagliati a lunghezza fissa. Questonotebook costruisce l'insieme di dati su cui lavora la seconda parte dell'indagine: tutti icuscinetti disponibili, tutte le condizioni operative, e frame tagliati sul giro meccanicovero invece che su un numero fisso di campioni.Tre scelte lo distinguono dal dataset della replica.**Ventinove cuscinetti invece di diciassette.** Ai sei sani e agli undici a danno reale siaggiungono i dodici a danno artificiale, prodotto a macchina su esemplari mai messi inesercizio. Restano fuori i tre KB, che hanno danni su entrambi gli anelli e non ricadono innessuna delle tre classi.**Il frame e un giro dell'albero, non 2560 campioni.** La velocita viene misurata su ogniregistrazione e il taglio segue quella, non un valore nominale. Poi ogni giro viene portatoalla lunghezza del giro piu lungo presente nel dataset, cosi si sovracampiona soltanto e nonsi introduce aliasing.**Il segnale resta in ampere.** Nessuna standardizzazione per frame: e la differenza piuimportante rispetto alla versione precedente di questo dataset. Dividere ogni frame per lapropria deviazione standard impone a tutti la stessa ampiezza, e l'ampiezza e proprio lagrandezza su cui la Fase 1 ha trovato l'unico miglioramento reale. Se servira una scala, sarauna costante unica applicata nel notebook del modello, non una divisione frame per frame.Il notebook si esegue una volta sola e lascia su Drive un file che i notebook successivileggono.

In [ ]:
!apt-get -qq update && apt-get -qq install -y unrar
!pip -q install requests scipy

In [ ]:
import os, sys, time, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO = 'https://github.com/matpaol/MacchineEdAzionamentiExam'
possibili = ['.', '..', '../codice',
             '/content/drive/MyDrive/MacchineEdAzionamentiExam',
             '/content/MacchineEdAzionamentiExam']
percorso_codice = None
for c in possibili:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break
if percorso_codice is None:
    print('codice non trovato in locale, clono il repo')
    subprocess.run(['git', 'clone', '-q', REPO, '/content/MacchineEdAzionamentiExam'], check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam'
sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='04_costruzione_dataset')

print('codice da', percorso_codice)
print('il dataset finira in', P['dataset'])

## I ventinove cuscinetti

In [ ]:
cuscinetti = config.CUSCINETTI_ESTESO

riepilogo = pd.DataFrame([{'cuscinetto': b,
                           'classe': config.NOMI_CLASSI[config.CLASSE_DI_ESTESO[b]],
                           'natura': config.NATURA_DI[b]} for b in cuscinetti])

print(len(cuscinetti), 'cuscinetti')
print(riepilogo.groupby(['classe', 'natura']).size().to_string())
print()
print('esclusi perche hanno danni su entrambi gli anelli:', ', '.join(config.CUSCINETTI_KB))
print()
print('estensione del danno dei dodici artificiali, secondo la norma VDI 3832:')
print('   livello 1, danno piccolo :', ', '.join(config.ESTENSIONE[1]))
print('   livello 2, danno grande  :', ', '.join(config.ESTENSIONE[2]))

## Scarico ed estrazioneGli archivi sono gia su Drive dalle esecuzioni precedenti, quindi la prima cella non scaricanulla. L'estrazione dei dodici cuscinetti artificiali, che i notebook della replica nonusavano, richiede qualche minuto la prima volta.

In [ ]:
archivi, dimensione = f.scarica_archivi(cuscinetti, P['raw'])
print(len(archivi), 'archivi su Drive |', round(dimensione / 1e9, 2), 'GB')

f.estrai_misure(cuscinetti, P['raw'], P['estratti'])
registrazioni = f.elenco_registrazioni(cuscinetti, P['estratti'])
print(len(registrazioni), 'registrazioni estratte |', 'attese', len(cuscinetti) * 20 * 4)

## Lettura, controlli e costruzione in un solo passaggioOgni registrazione viene letta una volta sola. Per ciascuna si misura la velocita dal canalededicato, si controlla la qualita del segnale, e se il controllo passa la si taglia in giri ela si porta alla lunghezza comune.I controlli sono quattro, e sono scelti in modo da colpire i guasti dello **strumento** e maiquelli del cuscinetto. Un filtro generico del tipo "scarta i valori anomali" eliminerebbeproprio i cuscinetti danneggiati, che dei valori anomali li producono per definizione. Ogniregola supera la prova: *un guasto al cuscinetto potrebbe produrre questo?* Se la risposta esi, la regola non e valida.| Regola | Soglia | Perche ||---|---|---|| file illeggibile | — | non si puo aprire || valori non finiti | oltre zero | dato mancante || canale piatto | deviazione nulla | sensore scollegato || sensore saturo | oltre l'1% dei campioni al fondo scala | l'ampiezza vera e fuori scala || velocita non stazionaria | variazione oltre l'1% della media | "un giro" non ha una lunghezza definita |Le soglie all'1% non sono scelte perche uno e un bel numero: sono difendibili perche ilrisultato non ne dipende. Sulla velocita, la registrazione patologica sta al 16,58% e laseconda peggiore allo 0,80%, quindi qualunque soglia fra 0,81% e 16,57% da lo stesso esito.Sulla saturazione, i dati puliti stanno allo 0,0016% e una saturazione vera al 73%. C'e ancheun criterio fisico: con lo 0,8% di variazione il confine del frame sbaglia di otto gradi,trascurabile; con il 16,6% sbaglia di novantasette, e il frame non e piu un giro.Tutte le regole sono relative ai valori del singolo file, mai assolute: cosi una registrazionea coppia ridotta e una a pieno carico vengono giudicate con lo stesso metro.La lunghezza di destinazione e fissata a priori al giro piu lungo atteso, cioe quello allavelocita piu bassa del banco. Alla fine si verifica che nessuna registrazione ne abbiarichiesto uno piu lungo.

In [ ]:
lunghezza = config.LUNGHEZZA_GIRO
massimo_giri_per_registrazione = 110          # il massimo osservabile e 108

X = np.empty((len(registrazioni) * massimo_giri_per_registrazione, lunghezza),
             dtype=np.float32)
righe_anagrafica = []
scansione = []
scartate = []
posizione = 0
partenza = time.time()

for k, percorso in enumerate(registrazioni, 1):
    m = f.metadati_nome(percorso)
    try:
        struct = f.apri(percorso)
        corrente = f.leggi(percorso, config.CANALE_CORRENTE, struct)
        velocita = f.leggi(percorso, 'speed', struct)
    except Exception as errore:
        scartate.append({**m, 'motivo': 'illeggibile: ' + type(errore).__name__})
        continue

    qualita = f.caratteristiche(corrente)
    moto = f.stazionarieta(velocita)
    scansione.append({**m, **qualita, 'rpm': moto['media'],
                      'variazione_velocita_pct': moto['variazione_pct']})

    if qualita['non_finiti'] > 0:
        scartate.append({**m, 'motivo': 'valori non finiti'})
        continue
    if qualita['std'] == 0:
        scartate.append({**m, 'motivo': 'canale piatto'})
        continue
    if qualita['al_fondo_scala'] / qualita['campioni'] > 0.01:
        scartate.append({**m, 'motivo': 'sensore saturo'})
        continue
    if moto['variazione_pct'] > 1.0:
        scartate.append({**m, 'motivo': 'velocita non stazionaria'})
        continue

    giri = f.porta_a_lunghezza(
        f.taglia_in_giri(corrente, f.campioni_per_giro(moto['media'])), lunghezza)
    quanti = len(giri)
    X[posizione:posizione + quanti] = giri
    for g in range(quanti):
        righe_anagrafica.append({'cuscinetto': m['cuscinetto'],
                                 'natura': config.NATURA_DI[m['cuscinetto']],
                                 'classe': config.CLASSE_DI_ESTESO[m['cuscinetto']],
                                 'regime': m['regime'],
                                 'registrazione': m['registrazione'],
                                 'giro': g})
    posizione = posizione + quanti

    if k % 400 == 0:
        print('  ', k, 'di', len(registrazioni), '|', posizione, 'frame |',
              round(time.time() - partenza), 's')

X = X[:posizione]
anagrafica = pd.DataFrame(righe_anagrafica)
scansione = pd.DataFrame(scansione)
scartate = pd.DataFrame(scartate)

print()
print('fatto in', round(time.time() - partenza), 's')
print('frame:', X.shape[0], '| lunghezza:', X.shape[1],
      '| memoria:', round(X.nbytes / 1e9, 2), 'GB')

## I controlli, cosa hanno trovato

In [ ]:
print('registrazioni lette:', len(scansione), '| scartate:', len(scartate))
if len(scartate):
    print()
    print(scartate[['registrazione', 'motivo']].to_string(index=False))
print()

print('valori non finiti in tutto il dataset:', int(scansione['non_finiti'].sum()))
print('canali piatti:', int((scansione['std'] == 0).sum()))
print()
print('qualita del segnale, su tutte le registrazioni:')
print(scansione[['rms', 'picco', 'crest', 'al_fondo_scala']].describe().loc[
    ['min', '50%', 'max']].round(3).to_string())
print()
print('stazionarieta della velocita, per regime:')
print(scansione.groupby('regime')[['rpm', 'variazione_velocita_pct']].agg(
    ['mean', 'max']).round(3).to_string())

In [ ]:
giro_richiesto = scansione['rpm'].apply(f.campioni_per_giro)
print('giro piu lungo richiesto:', int(giro_richiesto.max()), 'campioni',
      '| lunghezza usata:', lunghezza)
print('nessuna registrazione ha richiesto un giro piu lungo:',
      bool(giro_richiesto.max() <= lunghezza))
print()
print('velocita minima misurata:', round(scansione['rpm'].min(), 1), 'rpm')
print()
print('ampiezza del segnale per regime, in ampere:')
print(scansione.groupby('regime')[['rms', 'picco']].mean().round(4).to_string())

## Cosa abbiamo ottenutoDue verifiche prima di salvare. La prima e che i frame siano davvero rimasti in ampere: se lamedia e la deviazione di ogni frame fossero zero e uno, la standardizzazione sarebbe stataapplicata per errore. La seconda e la composizione dell'insieme, che condiziona come sileggeranno le metriche piu avanti.

In [ ]:
campione = X[np.random.default_rng(0).choice(len(X), 2000, replace=False)]
print('media dei frame:      da', round(float(campione.mean(axis=1).min()), 4),
      'a', round(float(campione.mean(axis=1).max()), 4))
print('deviazione dei frame: da', round(float(campione.std(axis=1).min()), 4),
      'a', round(float(campione.std(axis=1).max()), 4))
# la standardizzazione per frame imporrebbe deviazione esattamente 1 a ognuno
standardizzato = bool(np.allclose(campione.std(axis=1), 1.0, atol=1e-3))
print('deviazione uguale a 1 per tutti i frame:', standardizzato)
print('il segnale e in ampere, non standardizzato:', not standardizzato)
print()

conteggi = anagrafica.groupby(['classe', 'natura']).size().unstack(fill_value=0)
conteggi.index = [config.NOMI_CLASSI[c] for c in conteggi.index]
print('frame per classe e natura del danno:')
print(conteggi.to_string())
print()
quote = anagrafica.groupby('classe').size() / len(anagrafica)
for c, q in quote.items():
    print('  ', config.NOMI_CLASSI[c], round(100 * q, 1), '%')
print()
print('frame per regime:')
print(anagrafica.groupby('regime').size().to_string())
print()
print('cuscinetti:', anagrafica['cuscinetto'].nunique(),
      '| registrazioni:', anagrafica['registrazione'].nunique())

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(12, 4))

per_cuscinetto = anagrafica.groupby('cuscinetto').size()
colori = [f.COLORI[config.NOMI_CLASSI[config.CLASSE_DI_ESTESO[b]]]
          for b in per_cuscinetto.index]
assi[0].bar(range(len(per_cuscinetto)), per_cuscinetto.values, color=colori)
assi[0].set_xticks(range(len(per_cuscinetto)))
assi[0].set_xticklabels(per_cuscinetto.index, rotation=90, fontsize=7)
assi[0].set_ylabel('frame')
assi[0].set_title('Frame disponibili per cuscinetto', fontsize=10)
for cl in config.NOMI_CLASSI:
    assi[0].plot([], [], 's', color=f.COLORI[cl], label=cl)
assi[0].legend(fontsize=7)

for regime, gruppo in scansione.groupby('regime'):
    assi[1].hist(gruppo['rms'], bins=40, alpha=0.65, label=regime)
assi[1].set_xlabel('valore efficace della corrente (A)')
assi[1].set_ylabel('registrazioni')
assi[1].set_title('Ampiezza del segnale nei quattro regimi', fontsize=10)
assi[1].legend(fontsize=7)

f.salva_figura(fig, 'composizione_dataset', P['figure'])
plt.show()

## SalvataggioLa matrice dei frame viene salvata come file singolo, cosi il notebook successivo puoleggerla a blocchi senza caricarla tutta in memoria. Accanto vanno l'anagrafica riga perriga, la scansione con tutti i controlli di qualita, e i parametri usati.

In [ ]:
np.save(os.path.join(P['dataset'], 'frame.npy'), X)
anagrafica.to_csv(os.path.join(P['dataset'], 'anagrafica.csv'), index=False)
scansione.to_csv(os.path.join(P['dataset'], 'scansione.csv'), index=False)
scartate.to_csv(os.path.join(P['dataset'], 'scartate.csv'), index=False)

parametri = {'cuscinetti': cuscinetti,
             'canale': config.CANALE_CORRENTE,
             'fs': config.FS_ATTESO,
             'lunghezza_giro': lunghezza,
             'standardizzazione_per_frame': False,
             'soglia_variazione_velocita_pct': 1.0,
             'frame': int(X.shape[0]),
             'registrazioni_usate': int(anagrafica['registrazione'].nunique()),
             'registrazioni_scartate': scartate.to_dict('records')}
with open(os.path.join(P['dataset'], 'parametri.json'), 'w') as fh:
    json.dump(parametri, fh, indent=2)

print('salvato in', P['dataset'])
for nome in sorted(os.listdir(P['dataset'])):
    percorso = os.path.join(P['dataset'], nome)
    if os.path.isfile(percorso):
        print('  ', nome, round(os.path.getsize(percorso) / 1e6), 'MB')